# AdS4 Phase-Lock Probe-Break Test (v8.2)

This notebook removes **direct metric supervision** and tests whether uniqueness truly emerges from stacked probes.

We compare:
- **full stack**: EE + WL + GEO + consistency
- **drop GEO**: remove one probe
- **corrupt GEO**: keep the probe but bias its target

Goal:
- demonstrate genuine degeneracy return when one probe is removed
- show drift under structured probe corruption
- isolate reconstruction from observables instead of ground-truth metric fitting


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


In [ ]:
# Ground-truth AdS4-style toy background (used only to generate observables + diagnostics)
r = torch.linspace(1.05, 3.5, 240, device=device).view(-1, 1)

def f_true(r):
    return r**2 + 1 - 1/r

def g_true(r):
    return 1/(f_true(r) + 1e-6)

def ee_true(r):
    return torch.log(1.0 + 0.6 * f_true(r))

def wl_true(r):
    return torch.sqrt(f_true(r) + 1.8 * g_true(r) + 1e-6)

def geo_true(r):
    return torch.sqrt(f_true(r) + 0.7 * g_true(r) + 1e-6)


In [ ]:
# Observational targets
ee_obs = ee_true(r).detach()
wl_obs = wl_true(r).detach()
geo_obs = geo_true(r).detach()

def structured_geo_target(r, corruption_strength=0.12):
    base = geo_true(r)
    radial_bias = 1.0 + corruption_strength * (0.30 * (r - r.min()) / (r.max() - r.min()))
    oscillation = 1.0 + corruption_strength * 0.15 * torch.sin(2.2 * r)
    return (base * radial_bias * oscillation).detach()


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = MLP()
        self.g = MLP()

    def forward(self, r):
        # positive learned metric functions, but not directly supervised
        f = F.softplus(self.f(r))
        g = F.softplus(self.g(r))
        return f, g


In [ ]:
def train(mode='full', epochs=2000, corruption_strength=0.12, consistency_weight=0.15):
    m = Model().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=2e-3)

    loss_hist = []
    consistency_hist = []
    ee_hist = []
    wl_hist = []
    geo_hist = []

    geo_target = geo_obs if mode != 'corrupt_geo' else structured_geo_target(r, corruption_strength)

    for _ in range(epochs):
        opt.zero_grad()
        f, g = m(r)

        # Reconstructed observables from learned metric functions
        ee_pred = torch.log(1.0 + 0.6 * f)
        wl_pred = torch.sqrt(f + 1.8 * g + 1e-6)
        geo_pred = torch.sqrt(f + 0.7 * g + 1e-6)

        loss_ee = ((ee_pred - ee_obs)**2).mean()
        loss_wl = ((wl_pred - wl_obs)**2).mean()
        loss_geo = ((geo_pred - geo_target)**2).mean() if mode != 'drop_geo' else torch.tensor(0.0, device=device)

        # Keep the learned geometry class coherent without directly fitting f/g to truth
        loss_consistency = ((f * g - 1.0)**2).mean()

        loss = loss_ee + loss_wl + loss_geo + consistency_weight * loss_consistency
        loss.backward()
        opt.step()

        loss_hist.append(loss.item())
        consistency_hist.append(loss_consistency.item())
        ee_hist.append(loss_ee.item())
        wl_hist.append(loss_wl.item())
        geo_hist.append(loss_geo.item() if mode != 'drop_geo' else 0.0)

    with torch.no_grad():
        f_pred, g_pred = m(r)
        ee_pred = torch.log(1.0 + 0.6 * f_pred)
        wl_pred = torch.sqrt(f_pred + 1.8 * g_pred + 1e-6)
        geo_pred = torch.sqrt(f_pred + 0.7 * g_pred + 1e-6)

        # Diagnostics against truth (not used in optimization)
        metric_err = ((f_pred - f_true(r)).abs() + (g_pred - g_true(r)).abs()) / 2.0

    return {
        'model': m,
        'loss_hist': loss_hist,
        'consistency_hist': consistency_hist,
        'ee_hist': ee_hist,
        'wl_hist': wl_hist,
        'geo_hist': geo_hist,
        'f_pred': f_pred,
        'g_pred': g_pred,
        'ee_pred': ee_pred,
        'wl_pred': wl_pred,
        'geo_pred': geo_pred,
        'geo_target': geo_target,
        'metric_err': metric_err,
    }


In [ ]:
run_full = train(mode='full')
run_drop = train(mode='drop_geo')
run_corrupt = train(mode='corrupt_geo', corruption_strength=0.12)


In [ ]:
# Fast overview: total loss only
plt.figure(figsize=(8,4))
plt.plot(run_full['loss_hist'], label='full stack')
plt.plot(run_drop['loss_hist'], label='drop GEO')
plt.plot(run_corrupt['loss_hist'], label='corrupt GEO')
plt.legend()
plt.title('probe-break loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.tight_layout()
plt.show()


In [ ]:
# Main diagnostic figure
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

axes[0].plot(r.cpu(), f_true(r).cpu(), label='f true')
axes[0].plot(r.cpu(), run_full['f_pred'].cpu(), '--', label='f full')
axes[0].plot(r.cpu(), run_drop['f_pred'].cpu(), ':', label='f drop GEO')
axes[0].set_title('f(r): degeneracy return')
axes[0].legend(fontsize=8)

axes[1].plot(r.cpu(), g_true(r).cpu(), label='g true')
axes[1].plot(r.cpu(), run_full['g_pred'].cpu(), '--', label='g full')
axes[1].plot(r.cpu(), run_drop['g_pred'].cpu(), ':', label='g drop GEO')
axes[1].set_title('g(r): probe removal')
axes[1].legend(fontsize=8)

axes[2].plot(run_full['loss_hist'], label='full')
axes[2].plot(run_drop['loss_hist'], label='drop GEO')
axes[2].plot(run_corrupt['loss_hist'], label='corrupt GEO')
axes[2].set_title('stack break threshold')
axes[2].legend(fontsize=8)

axes[3].plot(r.cpu(), geo_obs.cpu(), label='GEO true')
axes[3].plot(r.cpu(), run_full['geo_pred'].cpu(), '--', label='full GEO pred')
axes[3].plot(r.cpu(), run_corrupt['geo_target'].cpu(), ':', label='corrupt GEO target')
axes[3].set_title('structured probe corruption')
axes[3].legend(fontsize=8)

axes[4].plot(r.cpu(), run_full['metric_err'].cpu(), label='full metric err')
axes[4].plot(r.cpu(), run_drop['metric_err'].cpu(), label='drop GEO err')
axes[4].plot(r.cpu(), run_corrupt['metric_err'].cpu(), label='corrupt GEO err')
axes[4].set_title('error widening')
axes[4].legend(fontsize=8)

axes[5].plot(run_full['consistency_hist'], label='full consistency')
axes[5].plot(run_drop['consistency_hist'], label='drop GEO consistency')
axes[5].plot(run_corrupt['consistency_hist'], label='corrupt GEO consistency')
axes[5].set_title('consistency drift')
axes[5].legend(fontsize=8)

plt.suptitle('AdS4 Phase-Lock Probe-Break Test (v8.2)')
plt.tight_layout()
plt.savefig('ads4_phase_lock_v8_2_probe_break.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: ads4_phase_lock_v8_2_probe_break.png')


In [ ]:
# Error diagnostics only
f_true_vals = f_true(r).detach()
g_true_vals = g_true(r).detach()

f_err = (run_full['f_pred'] - f_true_vals).abs()
g_err = (run_full['g_pred'] - g_true_vals).abs()
f_rel = f_err / (f_true_vals.abs() + 1e-6)
g_rel = g_err / (g_true_vals.abs() + 1e-6)

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(r.cpu(), f_err.cpu(), label='f abs err')
plt.plot(r.cpu(), g_err.cpu(), label='g abs err')
plt.title('absolute error (full stack)')
plt.xlabel('r')
plt.legend()

plt.subplot(1,2,2)
plt.plot(r.cpu(), f_rel.cpu(), label='f rel err')
plt.plot(r.cpu(), g_rel.cpu(), label='g rel err')
plt.title('relative error (full stack)')
plt.xlabel('r')
plt.legend()

plt.tight_layout()
plt.savefig('ads4_phase_lock_v8_2_probe_break_errors.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: ads4_phase_lock_v8_2_probe_break_errors.png')
